# LLM-based factual / semantic content preservation

**Goal:** build a fine-grained `L_facts` complementary to the existing TPI, semantic similarity and NLI metrics.

The evaluator is intentionally **not** a single "are these texts equivalent?" classifier. It uses two stages:

1. **Atomic semantic fact extraction** from the original transcript.
2. **Fact verification** against the anonymized transcript.

Each original fact is classified as:

- `PRESERVED`
- `ALTERED`
- `CONTRADICTED`
- `OMITTED`
- `UNCERTAIN`

The notebook then aggregates these judgments into a transparent `L_facts`.

> **Important research design:** the LLM is a semantic-information extractor/judge. `L_facts` is a deterministic aggregation of its structured judgments. The weights below are provisional and should **not** be presented as validated until compared against human annotations.

The implementation is designed for the current VoicePrivacy original/anonymized transcript pairs, but the input path is configurable.


## Why this design?

A global embedding score can remain high when one important fact changes. A global NLI score can also be noisy. Here we explicitly ask:

> **Which semantic information present in the original is still true in the anonymized transcript?**

The fact schema is deliberately richer than simple `(subject, relation, object)` triples. Facts can encode:

- entities
- events/actions
- attributes
- quantities/numbers
- temporal information
- locations
- negation
- modality
- causal relations
- comparisons

This makes the score directional:

\[
O \rightarrow A
\]

where `O` is the original and `A` the anonymized transcript.


In [1]:
# If needed on the HPC:
# !pip install -U transformers accelerate torch pandas tqdm pydantic

# Optional for a vLLM-based deployment:
# !pip install -U openai


In [2]:
import os
import re
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

pd.set_option("display.max_colwidth", 200)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0
CUDA available: False


## 1. Configuration

For the first experiment, **do not run all 100 pairs immediately**.

Start with 5–10 pairs, inspect the JSON outputs manually, refine the prompt/schema, then scale to 30–100 pairs.

The default model below is `Qwen/Qwen3-30B-A3B-Instruct-2507`. The official model card supports local Transformers inference and vLLM deployment. 


In [3]:
MODEL_ID = "Qwen/Qwen3-30B-A3B-Instruct-2507"

# Put your existing transcript/metrics CSV here.
# The notebook tries to detect common column names automatically.
INPUT_CSV = "vpc2026_b2_100_content_metrics.csv"

OUTPUT_DIR = Path("facts_evaluation")
OUTPUT_DIR.mkdir(exist_ok=True)

FACTS_JSONL = OUTPUT_DIR / "fact_judgments.jsonl"
RESULTS_CSV = OUTPUT_DIR / "facts_content_metrics.csv"

# First run: keep this small.
MAX_ROWS = 10

# Generation settings: deterministic evaluation.
MAX_NEW_TOKENS_EXTRACT = 1200
MAX_NEW_TOKENS_VERIFY = 1800
TEMPERATURE = 0.0

# Provisional loss weights.
# These are deliberately exposed so they can later be calibrated against humans.
STATUS_LOSS = {
    "PRESERVED": 0.0,
    "ALTERED": 0.5,
    "CONTRADICTED": 1.0,
    "OMITTED": 1.0,
    "UNCERTAIN": 0.5,
}

print("Model:", MODEL_ID)
print("Input:", INPUT_CSV)
print("Output:", OUTPUT_DIR)


Model: Qwen/Qwen3-30B-A3B-Instruct-2507
Input: vpc2026_b2_100_content_metrics.csv
Output: facts_evaluation


In [4]:
def find_text_columns(df):
    candidates = [
        ("original_text", "anonymized_text"),
        ("original_transcript", "anonymized_transcript"),
        ("transcript_original", "transcript_anonymized"),
        ("text_original", "text_anonymized"),
        ("orig_text", "anon_text"),
    ]
    for a, b in candidates:
        if a in df.columns and b in df.columns:
            return a, b
    return None, None


if Path(INPUT_CSV).exists():
    df = pd.read_csv(INPUT_CSV)
    orig_col, anon_col = find_text_columns(df)
    print("Loaded:", len(df), "rows")
    print("Detected columns:", orig_col, anon_col)
    if orig_col is None:
        print("Columns available:", df.columns.tolist())
        raise ValueError(
            "Could not automatically detect original/anonymized transcript columns. "
            "Set orig_col and anon_col manually."
        )
else:
    print("Input CSV not found in the current directory.")
    print("Using a tiny demo dataset so that the notebook can be tested immediately.")
    df = pd.DataFrame({
        "pair_id": ["demo_1", "demo_2", "demo_3"],
        "original_text": [
            "John moved to London in 2020 because he got a new job.",
            "The company opened a factory in Geneva with 200 employees.",
            "She did not attend the meeting because she was ill."
        ],
        "anonymized_text": [
            "John moved to London in 2021 because he got a new job.",
            "The company opened a factory in Lausanne with 200 employees.",
            "She attended the meeting because she was ill."
        ]
    })
    orig_col, anon_col = "original_text", "anonymized_text"

work_df = df.copy()
work_df = work_df[[c for c in work_df.columns if c in df.columns]].copy()
work_df["original_text"] = work_df[orig_col].fillna("").astype(str)
work_df["anonymized_text"] = work_df[anon_col].fillna("").astype(str)

if MAX_ROWS is not None:
    work_df = work_df.head(MAX_ROWS).copy()

print("Rows to evaluate:", len(work_df))
display(work_df[["original_text", "anonymized_text"]].head())


Input CSV not found in the current directory.
Using a tiny demo dataset so that the notebook can be tested immediately.
Rows to evaluate: 3


,original_text,anonymized_text
0,John moved to London in 2020 because he got a new job.,John moved to London in 2021 because he got a new job.
1,The company opened a factory in Geneva with 200 employees.,The company opened a factory in Lausanne with 200 employees.
2,She did not attend the meeting because she was ill.,She attended the meeting because she was ill.


## 2. Load Qwen

The 30B-A3B model is a MoE model but its repository is still large. On an HPC node, `device_map="auto"` lets Transformers distribute the model over available GPUs.

If the node cannot hold the model, use a vLLM deployment or switch `MODEL_ID` to a quantized Qwen checkpoint.


In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("Loaded:", MODEL_ID)
print("Input device:", next(model.parameters()).device)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 16 files:   0%|                                | 0/16 [00:00<?, ?it/s]

## 3. Robust JSON generation

The model is instructed to return JSON only. The parser also handles accidental Markdown code fences.

For a research evaluation, **invalid outputs are not silently discarded**: they are marked as failures so that parsing reliability can be reported.


In [ ]:
def extract_json(text):
    text = text.strip()

    # Remove common markdown fences.
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    # First attempt: entire response.
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Second attempt: find the outermost JSON object.
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        candidate = text[start:end + 1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass

    raise ValueError("Could not parse valid JSON from model output.")


def qwen_generate(messages, max_new_tokens):
    # Qwen3 chat template supports disabling thinking for concise structured outputs.
    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        )
    except TypeError:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

    # With device_map="auto", inputs should go to the model's input device.
    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## 4. Stage A — atomic semantic fact extraction

The extraction prompt asks for **minimal but meaningful semantic units**.

A good fact should be something whose truth value can be checked independently.

Bad:
> "John moved to London in 2020 because he got a new job."

Good:
- John moved to London.
- The move happened in 2020.
- John got a new job.
- The new job caused the move.

The model is also asked to preserve exact entities, numbers, dates and negation because those are especially important for content preservation.


In [ ]:
EXTRACTION_SYSTEM = r'''
You are a semantic information extraction system for evaluating speech anonymization.

Your task is to decompose the ORIGINAL transcript into atomic semantic facts whose preservation
can be checked against another transcript.

A fact must represent information that matters to the meaning of the utterance.

Include, when present:
- people, organizations, places and other entities;
- events/actions and their participants;
- attributes;
- quantities and numbers;
- dates and temporal relations;
- locations;
- negation;
- modality (possible, necessary, should, might, etc.);
- causal relations;
- comparisons;
- other explicit semantic relations.

Do NOT invent facts or use world knowledge.
Only extract information explicitly expressed in the ORIGINAL transcript.

Make facts as atomic as reasonably possible, but do not split a single inseparable meaning
into meaningless fragments.

Return ONLY valid JSON with this schema:

{
  "facts": [
    {
      "fact_id": "F1",
      "statement": "...",
      "type": "entity|event|attribute|quantity|temporal|location|negation|modality|causal|comparison|relation|other",
      "subject": "...",
      "predicate": "...",
      "object": "...",
      "attributes": {}
    }
  ]
}
'''.strip()


def extract_facts(text):
    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM},
        {
            "role": "user",
            "content": f"ORIGINAL TRANSCRIPT:\n{text}\n\nExtract the atomic semantic facts."
        }
    ]

    raw = qwen_generate(messages, MAX_NEW_TOKENS_EXTRACT)
    parsed = extract_json(raw)

    if "facts" not in parsed or not isinstance(parsed["facts"], list):
        raise ValueError("Extraction JSON does not contain a 'facts' list.")

    return parsed, raw


### Test extraction on one example


In [ ]:
demo_text = work_df.iloc[0]["original_text"]
facts_demo, raw_demo = extract_facts(demo_text)

print(json.dumps(facts_demo, indent=2, ensure_ascii=False))


## 5. Stage B — verify each original fact against the anonymized transcript

This is the central judge.

Crucially, the model is **not** asked whether the entire texts are equivalent. It receives the extracted facts from the original and verifies them one by one against the anonymized transcript.

`ALTERED` is used when the same underlying fact is present but an important value changed, e.g. `2020 → 2021`.

`CONTRADICTED` is reserved for an explicit semantic conflict.

`OMITTED` means the original fact is no longer supported by the anonymized transcript, without enough evidence to call it a contradiction.

`UNCERTAIN` captures cases where the transcript is too ambiguous or noisy to make a reliable decision.


In [ ]:
VERIFY_SYSTEM = r'''
You are a rigorous semantic fact verifier evaluating an anonymized transcript.

You receive:
1. an ORIGINAL transcript;
2. an ANONYMIZED transcript;
3. atomic semantic facts extracted from the ORIGINAL.

For EACH original fact, determine whether the fact remains true in the ANONYMIZED transcript.

Use exactly one status:

PRESERVED:
The anonymized transcript still expresses the same fact with no meaningful semantic loss.

ALTERED:
The same fact/event is present but an important semantic value changed
(e.g. number, date, entity, location, attribute, degree, participant).

CONTRADICTED:
The anonymized transcript explicitly states something incompatible with the original fact.

OMITTED:
The original fact is no longer expressed/supported in the anonymized transcript,
but there is no explicit contradictory statement.

UNCERTAIN:
The available transcript evidence is insufficient or ambiguous.

Important rules:
- Do not use world knowledge.
- Do not penalize harmless paraphrasing.
- Do not treat lexical differences as semantic changes.
- Preserve exact distinctions in numbers, dates, entities, negation, modality, causality and relations.
- Be conservative: if the transcript does not support a fact, do not assume it.
- Explain the evidence using the ANONYMIZED transcript only.
- Do not create new original facts.

Return ONLY valid JSON:

{
  "verifications": [
    {
      "fact_id": "F1",
      "status": "PRESERVED|ALTERED|CONTRADICTED|OMITTED|UNCERTAIN",
      "confidence": 0.0,
      "evidence": "...",
      "change_type": "none|entity|event|attribute|quantity|temporal|location|negation|modality|causal|comparison|relation|other",
      "change_description": "..."
    }
  ]
}
'''.strip()


def verify_facts(original, anonymized, facts):
    facts_json = json.dumps(facts, ensure_ascii=False, indent=2)

    user_prompt = f'''
ORIGINAL TRANSCRIPT:
{original}

ANONYMIZED TRANSCRIPT:
{anonymized}

ATOMIC FACTS EXTRACTED FROM ORIGINAL:
{facts_json}

Verify every fact against the anonymized transcript.
'''.strip()

    messages = [
        {"role": "system", "content": VERIFY_SYSTEM},
        {"role": "user", "content": user_prompt},
    ]

    raw = qwen_generate(messages, MAX_NEW_TOKENS_VERIFY)
    parsed = extract_json(raw)

    if "verifications" not in parsed or not isinstance(parsed["verifications"], list):
        raise ValueError("Verification JSON does not contain a 'verifications' list.")

    return parsed, raw


## 6. Compute `L_facts`

For now, the score is deliberately transparent:

- `PRESERVED` = 0 loss
- `ALTERED` = 0.5
- `CONTRADICTED` = 1
- `OMITTED` = 1
- `UNCERTAIN` = 0.5

This is **not yet a validated loss**. These values are experimental defaults.

The main output should therefore include both:

1. the raw fact-level judgments;
2. the aggregated score.

Later, human annotations can be used to calibrate severity and possibly learn the aggregation.


In [ ]:
def compute_l_facts(verifications, status_loss=STATUS_LOSS):
    if not verifications:
        return np.nan

    losses = []
    for v in verifications:
        status = str(v.get("status", "UNCERTAIN")).upper()
        losses.append(status_loss.get(status, status_loss["UNCERTAIN"]))

    return float(np.mean(losses))


def fact_summary(verifications):
    counts = pd.Series(
        [str(v.get("status", "UNCERTAIN")).upper() for v in verifications]
    ).value_counts()

    return {
        "n_facts": len(verifications),
        "n_preserved": int(counts.get("PRESERVED", 0)),
        "n_altered": int(counts.get("ALTERED", 0)),
        "n_contradicted": int(counts.get("CONTRADICTED", 0)),
        "n_omitted": int(counts.get("OMITTED", 0)),
        "n_uncertain": int(counts.get("UNCERTAIN", 0)),
    }


## 7. Run the full pipeline on a small sample

Results are cached in JSONL. If the notebook is interrupted, completed pairs can be reused.

The cache stores the model outputs themselves, not just the final score, so individual decisions can be audited.


In [ ]:
def load_cache(path):
    cache = {}
    if Path(path).exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    obj = json.loads(line)
                    cache[str(obj["row_id"])] = obj
    return cache


cache = load_cache(FACTS_JSONL)
print("Cached evaluations:", len(cache))


In [ ]:
all_results = []

with open(FACTS_JSONL, "a", encoding="utf-8") as fout:
    for idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
        row_id = str(row.get("pair_id", idx))

        if row_id in cache:
            all_results.append(cache[row_id])
            continue

        original = row["original_text"]
        anonymized = row["anonymized_text"]

        try:
            extracted, raw_extract = extract_facts(original)
            verified, raw_verify = verify_facts(
                original,
                anonymized,
                extracted["facts"],
            )

            summary = fact_summary(verified["verifications"])
            l_facts = compute_l_facts(verified["verifications"])

            result = {
                "row_id": row_id,
                "original_text": original,
                "anonymized_text": anonymized,
                "facts": extracted["facts"],
                "verifications": verified["verifications"],
                "L_facts": l_facts,
                **summary,
                "extraction_raw": raw_extract,
                "verification_raw": raw_verify,
                "error": None,
            }

        except Exception as e:
            result = {
                "row_id": row_id,
                "original_text": original,
                "anonymized_text": anonymized,
                "facts": [],
                "verifications": [],
                "L_facts": np.nan,
                "n_facts": 0,
                "n_preserved": 0,
                "n_altered": 0,
                "n_contradicted": 0,
                "n_omitted": 0,
                "n_uncertain": 0,
                "extraction_raw": None,
                "verification_raw": None,
                "error": repr(e),
            }

        fout.write(json.dumps(result, ensure_ascii=False) + "\n")
        fout.flush()
        all_results.append(result)

results_df = pd.DataFrame([
    {
        k: v
        for k, v in r.items()
        if k not in {"facts", "verifications", "extraction_raw", "verification_raw"}
    }
    for r in all_results
])

display(results_df)


## 8. Inspect individual judgments

This is essential. Do not trust `L_facts` before reading the fact-level outputs.

The useful research object is not just a scalar score. It is an **auditable explanation of what semantic information the model thinks changed**.


In [ ]:
def show_case(result, max_facts=30):
    print("=" * 100)
    print("PAIR:", result["row_id"])
    print("\nORIGINAL:")
    print(result["original_text"])
    print("\nANONYMIZED:")
    print(result["anonymized_text"])
    print("\nL_facts:", result["L_facts"])

    facts = result.get("facts", [])
    verifications = result.get("verifications", [])

    vmap = {v.get("fact_id"): v for v in verifications}

    rows = []
    for f in facts[:max_facts]:
        v = vmap.get(f.get("fact_id"), {})
        rows.append({
            "fact_id": f.get("fact_id"),
            "type": f.get("type"),
            "fact": f.get("statement"),
            "status": v.get("status"),
            "confidence": v.get("confidence"),
            "change_type": v.get("change_type"),
            "evidence": v.get("evidence"),
            "change": v.get("change_description"),
        })

    display(pd.DataFrame(rows))


for r in all_results[:3]:
    show_case(r)


## 9. Merge with the existing TPI / similarity / NLI metrics

If the input CSV already contains the metrics from notebook 09, this section lets us test whether `L_facts` adds information rather than simply reproducing the existing metrics.

The exact column names are detected where possible.


In [ ]:
# Try to identify the common notebook-09 metric columns.
metric_candidates = {
    "semantic_similarity": ["semantic_similarity", "similarity"],
    "TPI": ["TPI", "tpi"],
    "nli_entailment": ["nli_entailment", "entailment"],
    "nli_contradiction": ["nli_contradiction", "contradiction"],
    "nli_neutral": ["nli_neutral", "neutral"],
}

metric_map = {}
for canonical, candidates in metric_candidates.items():
    for c in candidates:
        if c in work_df.columns:
            metric_map[canonical] = c
            break

print("Detected metrics:", metric_map)

base_cols = []
if "pair_id" in work_df.columns:
    base_cols.append("pair_id")

merge_base = work_df.copy()
if "pair_id" in merge_base.columns:
    merge_base["row_id"] = merge_base["pair_id"].astype(str)
else:
    merge_base["row_id"] = merge_base.index.astype(str)

merge_base = merge_base[["row_id"] + list(metric_map.values())].rename(
    columns={v: k for k, v in metric_map.items()}
)

comparison_df = results_df.merge(merge_base, on="row_id", how="left")

display(comparison_df.head())


In [ ]:
# Correlation with the existing metrics, if available.
corr_cols = [
    c for c in [
        "L_facts",
        "semantic_similarity",
        "TPI",
        "nli_entailment",
        "nli_contradiction",
        "nli_neutral",
    ]
    if c in comparison_df.columns
]

if len(corr_cols) >= 2:
    display(comparison_df[corr_cols].corr(numeric_only=True).round(3))
else:
    print("Not enough existing metric columns detected for correlation analysis.")


## 10. Recommended next experiment: adversarial / controlled validation

Before using this as a real content loss, create controlled transformations where the expected semantic effect is known:

1. identical text
2. harmless paraphrase
3. synonym substitution
4. entity substitution
5. number substitution
6. date substitution
7. negation insertion/removal
8. event substitution
9. causal relation substitution
10. fact deletion
11. fact addition
12. generalization (`three dogs` → `dogs`)
13. contradiction
14. sentence reordering

The critical question is not whether `L_facts` correlates with similarity.

It is:

> **Does `L_facts` reliably increase when semantically important information is actually altered, while remaining low for meaning-preserving paraphrases?**

After that, validate a subset with human annotations.


## 11. Important limitations

### LLM-as-judge risk
The Qwen model can make extraction and verification errors. Its output is not ground truth.

### Prompt sensitivity
Changing the prompt can change fact granularity and therefore the loss. Keep the prompt versioned.

### Fact granularity
One model may extract 4 facts from a sentence while another extracts 7. This is a major methodological issue and must be evaluated.

### ASR confounding
The transcripts are produced by ASR. A fact change may come from transcription error rather than anonymization. Where possible, compare against reference transcripts or manually flag ASR-induced changes.

### Provisional loss weights
The current status weights are placeholders. Do not report them as scientifically validated.

### Leakage / circularity
If the same LLM is later used to create human-like labels and to evaluate them, avoid using those labels as an unquestioned ground truth.


## 12. Outputs

The notebook writes:

- `facts_evaluation/fact_judgments.jsonl` — auditable extraction + verification outputs;
- `facts_evaluation/facts_content_metrics.csv` — flat per-pair results.

The JSONL is the important artifact for research because it lets us inspect **which facts were judged preserved, altered, contradicted or omitted**.


In [ ]:
results_df.to_csv(RESULTS_CSV, index=False)
print("Saved:", RESULTS_CSV)
print("Saved:", FACTS_JSONL)
